# 자료탐색방법론 4주차: 줄기와 잎 그림, 히스토그램

교육용 가상 자료입니다. 위에서 아래로 실행하며 진단용 오류 코드와 수정 결과를 비교하세요.

[강의 슬라이드](https://datathinkinglab.github.io/kyonggi-courses-2026/eda/slides/week-04.html)


## 수업 목표


<callout icon="🎯" color="blue_bg">
	**오늘의 목표**
	1. 분포의 중심 부근·퍼짐·비대칭·빈 구간·큰 값을 그림에서 설명한다.
	2. 줄기와 잎 그림을 손으로 만들고 원자료와 대조한다.
	3. 히스토그램의 구간 경계와 빈도를 직접 계산한다.
	4. 같은 자료에서 구간 폭·시작점·표시 범위를 바꿔 결과를 비교한다.
	5. 빈도 합계·경계값·대안 계산으로 그림을 검증하고 해석의 한계를 기록한다.
</callout>

## 지난 시간에서 이어지는 질문


1주차에는 숫자 하나가 자료의 구조를 감출 수 있다는 점을 배웠고, 2주차에는 분석 명세를 작성했습니다. 3주차에는 사용한 행·분모·결측 처리와 계산 결과를 검증했습니다. 이번 주 질문은 **“그림도 맞는지 확인해야 하지 않을까?”**입니다.
그림이 깔끔하다는 것은 자료를 빠짐없이 표현했다는 증거가 아닙니다. 구간 폭이 너무 넓으면 구조를 감추고, 너무 좁으면 우연한 들쭉날쭉함을 강조할 수 있습니다. 그림을 만든 설정도 분석 명세의 일부입니다.

## 강의 운영안 · 3시간 배정


<table header-row="true">
<tr><td>시간</td><td>활동</td><td>학생 산출물</td></tr>
<tr><td>0–20분</td><td>EDA의 현시성 복습, 분포 읽기</td><td>그림에서 물어볼 질문 3개</td></tr>
<tr><td>20–40분</td><td>줄기와 잎 그림 원리와 손그림</td><td>키·단위가 있는 줄기와 잎 그림</td></tr>
<tr><td>40–60분</td><td>히스토그램의 경계·빈도·폭</td><td>손으로 만든 빈도표와 스케치</td></tr>
<tr><td>60–100분</td><td>AI와 코드로 손그림 재현</td><td>원자료 대조 및 빈도 검증</td></tr>
<tr><td>100–140분</td><td>구간 폭·시작점·범위 변화 실험</td><td>동일 자료의 비교 그림과 설정표</td></tr>
<tr><td>140–160분</td><td>경계값·결측·단일값 진단</td><td>예상값과 실제값을 적은 검증 로그</td></tr>
<tr><td>160–180분</td><td>해석·토론·마무리</td><td>관찰·추측·한계를 구분한 설명</td></tr>
</table>

시간은 활동 배분 예시이며 휴식과 토론 속도에 맞춰 조정합니다. 60분 개념·손계산과 120분 실습·검증·토론의 흐름입니다.

## 분석 자료와 질문을 먼저 정하기


이번에는 분포를 관찰하기 위한 **새로운 교육용 가상 통학 응답 22건**을 사용합니다. 2·3주차의 8명과 같은 학생들의 추가 기록이 아닙니다. 각 행은 서로 다른 가상 학생 한 명의 평소 편도 통학 응답입니다.

- 변수: minutes, 편도 통학 시간, 분 단위 비율척도.
- 정수는 이번 자료에 기록된 측정 정밀도이며 실제 시간이 언제나 정수라는 뜻은 아닙니다.
- 결측 2건은 미응답입니다. 원자료를 보존하고 그림에서는 유효 시간 20건만 사용합니다.
- 90분은 입력된 관측값입니다. 크다는 이유만으로 삭제하거나 오류로 단정하지 않습니다.
- 분석 질문: “관측된 통학 시간은 어디에 몰려 있고, 얼마나 퍼져 있으며, 큰 값 쪽으로 꼬리가 있는가?”

새 Colab 노트북을 만들고 아래 코드를 순서대로 실행하세요. 외부 파일이나 유료 AI 연결은 필요하지 않습니다.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

raw = pd.Series(
    [10, 12, 15, 18, 20, 20, 22, 25, 28, 30,
     32, 35, 38, 40, 42, 45, 50, 55, 60, 90, None, None],
    name="minutes", dtype="float64"
)
valid = raw.dropna()
assert np.isfinite(valid).all()
assert (valid >= 0).all()
assert (valid % 1 == 0).all()
x = valid.astype(int).to_numpy()
print("전체:", len(raw), "유효:", len(x), "결측:", raw.isna().sum())
print("정렬:", np.sort(x))

확인값은 전체 22건·유효 20건·결측 2건, 최솟값 10분·최댓값 90분입니다. 정수 변환은 정수값인지 확인한 뒤에만 했습니다. 소수 자료를 그대로 정수로 바꾸면 원자료가 잘릴 수 있습니다.

## 분포를 읽는 다섯 가지 질문


1. **어디에 몰려 있는가?** 값이 많이 모인 구간을 단위와 함께 말합니다.
2. **얼마나 퍼져 있는가?** 전체 범위와 대부분의 값이 모인 범위를 구분합니다.
3. **양쪽 모양이 비슷한가?** 긴 꼬리가 어느 쪽인지 관찰합니다.
4. **봉우리나 빈 구간이 있는가?** 구간 설정을 바꿔도 비슷하게 보이는지 확인합니다.
5. **떨어진 값이 있는가?** 수집 맥락과 원기록을 확인할 대상으로 남깁니다.

“오른쪽 꼬리가 보인다”는 관찰과 “정규분포가 아니다”라는 판정은 다릅니다. 작은 자료의 그림 하나만으로 모집단 분포를 확정하지 않습니다. 확률분포와 정규확률플롯은 10·11주차에서 다룹니다.

## 손으로 만드는 순서


1. 유효값 20개를 작은 순서로 정렬합니다.
2. 십의 자리를 줄기, 일의 자리를 잎으로 정합니다.
3. 같은 줄기의 잎을 작은 순서로 적고, 반복값도 반복해서 적습니다.
4. 줄기 1부터 9까지 빈 줄기도 유지합니다.
5. **키: 2 | 5 = 25분, 잎 단위=1분**을 반드시 표시합니다.

```plain text
줄기 | 잎
  1  | 0 2 5 8
  2  | 0 0 2 5 8
  3  | 0 2 5 8
  4  | 0 2 5
  5  | 0 5
  6  | 0
  7  |
  8  |
  9  | 0

키: 2 | 5 = 25분
```

잎을 모두 세면 20개입니다. 줄기 2의 잎 0이 두 번 나오는 것은 20분 응답이 두 건이기 때문입니다. 줄기 7·8을 생략하면 60분과 90분 사이의 간격이 덜 드러납니다.

## 코드로 재현하고 원자료를 복원하기


In [ ]:
stems = {}
for value in sorted(x.tolist()):
    stem, leaf = divmod(value, 10)
    stems.setdefault(stem, []).append(leaf)

for stem in range(min(stems), max(stems) + 1):
    leaves = stems.get(stem, [])
    print(f"{stem:2d} | " + " ".join(map(str, leaves)))

reconstructed = [
    10 * stem + leaf
    for stem in sorted(stems)
    for leaf in stems[stem]
]
assert reconstructed == sorted(x.tolist())
print("잎 수:", len(reconstructed), "원자료 복원 일치")

divmod는 몫과 나머지를 구합니다. 이 코드는 **비음수 정수이고 잎 단위가 1인 이번 자료**를 위한 것입니다. 음수·소수 자료에는 표시 규칙을 새로 정해야 합니다. 큰 자료에서는 잎이 너무 길어질 수 있어 히스토그램이 더 읽기 쉬울 수 있습니다.
줄기와 잎 그림은 정해진 정밀도의 값을 보존하지만, 히스토그램은 같은 구간 안의 정확한 값을 보여주지 않습니다. 두 그림의 역할을 구분하세요.

## 막대그래프와 무엇이 다른가?


교통수단별 인원을 보여주는 막대그래프는 범주를 비교합니다. 히스토그램은 수치형 변수의 연속된 구간을 비교합니다. 가로축은 분 단위 구간, 세로축은 이번 실습에서는 관측값의 개수입니다. 구간의 위치·폭에는 수치적 의미가 있습니다.

## 손계산 빈도표


먼저 경계를 0, 20, 40, 60, 80, 100분으로 정합니다. NumPy와 Matplotlib에서는 마지막 구간을 제외하고 왼쪽 경계는 포함하고 오른쪽은 제외합니다. 마지막 구간은 오른쪽 끝도 포함합니다. [NumPy 공식 문서](https://numpy.org/doc/stable/reference/generated/numpy.histogram.html)

<table header-row="true">
<tr><td>구간(분)</td><td>포함된 값</td><td>빈도</td><td>상대빈도</td></tr>
<tr><td>0 이상 20 미만</td><td>10, 12, 15, 18</td><td>4</td><td>0.20</td></tr>
<tr><td>20 이상 40 미만</td><td>20, 20, 22, 25, 28, 30, 32, 35, 38</td><td>9</td><td>0.45</td></tr>
<tr><td>40 이상 60 미만</td><td>40, 42, 45, 50, 55</td><td>5</td><td>0.25</td></tr>
<tr><td>60 이상 80 미만</td><td>60</td><td>1</td><td>0.05</td></tr>
<tr><td>80 이상 100 이하</td><td>90</td><td>1</td><td>0.05</td></tr>
<tr><td>합계</td><td>유효 응답 전체</td><td>20</td><td>1.00</td></tr>
</table>

**손그림 활동:** 가로축에 위 경계를 적고 높이 4·9·5·1·1의 직사각형을 그리세요. 20분과 40분이 어느 구간에 들어가는지 짝에게 설명합니다. 상대빈도의 분모는 결측을 포함한 22가 아니라 유효 응답 20입니다.

## 기준 코드


In [ ]:
edges = np.array([0, 20, 40, 60, 80, 100])
counts, returned_edges = np.histogram(x, bins=edges)
display(pd.DataFrame({
    "left": edges[:-1],
    "right": edges[1:],
    "count": counts,
    "relative_frequency": counts / len(x)
}))
assert counts.tolist() == [4, 9, 5, 1, 1]
assert counts.sum() == len(x)

fig, ax = plt.subplots(figsize=(8, 4))
plotted_counts, plotted_edges, _ = ax.hist(
    x, bins=edges, color="#2764a5", edgecolor="white"
)
ax.set(xlabel="Commute time (minutes)", ylabel="Count",
       title="Fictional commute data: valid n=20, missing=2",
       xlim=(0, 100), ylim=(0, 10))
ax.set_xticks(edges)
assert np.array_equal(plotted_counts, counts)
assert np.array_equal(plotted_edges, returned_edges)
plt.tight_layout()
plt.show()

축 라벨은 Colab의 한글 폰트 설정 없이도 읽을 수 있도록 영어를 사용했습니다. 그림을 보고 유효 응답 20건 중 20–40분 미만 구간에 9건이 있다는 것을 확인하세요.

## 실험 A · 구간 폭만 바꾸기


같은 x와 범위 0–100분을 유지하고 폭을 10·20·25분으로 바꿉니다. 가로축과 세로축 범위를 맞춰 나란히 비교합니다.

In [ ]:
widths = [10, 20, 25]
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharex=True, sharey=True)
for ax, width in zip(axes, widths):
    trial_edges = np.arange(0, 101, width)
    trial_counts, _ = np.histogram(x, bins=trial_edges)
    assert trial_counts.sum() == len(x)
    ax.hist(x, bins=trial_edges, color="#2764a5", edgecolor="white")
    ax.set(title=f"Width={width} min", xlabel="Minutes",
           xlim=(0, 100), ylim=(0, 11))
axes[0].set_ylabel("Count")
plt.tight_layout()
plt.show()

비교 기록:

- 폭 10분에서는 빈 구간이나 값의 드문 부분이 어떻게 보이는가?
- 폭 25분에서는 서로 다른 값들이 어떤 덩어리로 합쳐지는가?
- 세 그림 모두에서 유지되는 관찰은 무엇인가?
- **폭이 다른 그림의 막대 높이를 그대로 밀집도의 차이로 비교하지 마세요.** 넓은 구간은 더 많은 값을 담을 수 있습니다.

“bins=10”은 폭이 10분이라는 뜻이 아니라 구간 개수가 10개라는 뜻입니다. 재현이 필요하면 경계 목록을 명시합니다.

## 실험 B · 폭은 같게, 시작점만 바꾸기


폭 20분을 유지하고 첫 경계를 0분과 5분으로 바꿉니다. 두 경계 집합 모두 모든 유효값을 포함합니다.

In [ ]:
edge_sets = [np.arange(0, 101, 20), np.arange(5, 106, 20)]
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharex=True, sharey=True)
for ax, trial_edges in zip(axes, edge_sets):
    trial_counts, _ = np.histogram(x, bins=trial_edges)
    print("경계:", trial_edges.tolist(), "빈도:", trial_counts.tolist())
    assert trial_counts.sum() == len(x)
    ax.hist(x, bins=trial_edges, color="#39876c", edgecolor="white")
    ax.set(title=f"Start={trial_edges[0]}, width=20",
           xlabel="Minutes", xlim=(0, 110), ylim=(0, 11))
axes[0].set_ylabel("Count")
plt.tight_layout()
plt.show()

첫 설정은 [4, 9, 5, 1, 1], 두 번째는 [7, 8, 4, 0, 1]입니다. 시작점만 바꿔도 빈 구간이 새로 보일 수 있습니다. 자료가 바뀐 것이 아니라 묶는 방식이 달라진 것입니다.

## 실험 C · 범위를 좁히면 사라지는 값


In [ ]:
short_counts, short_edges = np.histogram(x, bins=[0, 20, 40, 60])
outside = x[(x < short_edges[0]) | (x > short_edges[-1])]
print("빈도:", short_counts.tolist())
print("그림에 포함된 수:", short_counts.sum())
print("범위 밖:", outside.tolist())
assert short_counts.sum() + len(outside) == len(x)

결과는 빈도 [4, 9, 6], 포함 19건, 범위 밖 [90]입니다. 마지막 구간에 60분이 포함됩니다. NumPy는 지정한 범위 밖의 값을 빈도에 넣지 않습니다. **명세 없이 90분을 숨긴 그림을 전체 자료의 분포로 제시하면 안 됩니다.**
표시용 xlim만 좁히는 것과 집계 경계 자체를 좁히는 것은 다릅니다. 전자는 화면 일부를 가리고, 후자는 실제 빈도 합계가 달라질 수 있습니다.

## ① 손계산 대조


잎 수 20, 구간 빈도 4·9·5·1·1, 빈도 합계 20, 상대빈도 합계 1을 확인합니다. 두 20분 응답을 한 건으로 합치지 않았는지도 봅니다.

## ② 다른 계산 경로로 교차확인


Matplotlib의 hist는 NumPy의 histogram을 이용하므로 두 결과가 같다는 것만으로 독립적 확인이 충분한 것은 아닙니다. 구간 조건을 직접 세는 반복문으로 확인합니다. [Matplotlib 공식 문서](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.hist.html)

In [ ]:
manual_counts = []
for i, (left, right) in enumerate(zip(edges[:-1], edges[1:])):
    last_bin = i == len(edges) - 2
    n = sum(
        1 for value in x
        if value >= left and (value <= right if last_bin else value < right)
    )
    manual_counts.append(n)

assert manual_counts == counts.tolist()
assert np.isclose((counts / len(x)).sum(), 1)
print("조건문 직접 집계:", manual_counts)

이 검사는 구간 계산 경로를 달리 확인합니다. 입력된 숫자가 실제 학생의 응답인지까지 확인해 주지는 않습니다.

## ③ 경계값·극단 사례 투입


실행 전에 각 결과를 종이에 예상합니다. 사례는 원자료와 분리합니다.

In [ ]:
test_cases = {
    "boundary": [0, 20, 40, 60, 80, 100],
    "ties": [20, 20, 20],
    "single": [20],
    "all_missing": [None, None],
    "outside": [10, 20, 150]
}
for name, values in test_cases.items():
    test = pd.Series(values, dtype="float64").dropna().to_numpy()
    c, _ = np.histogram(test, bins=edges)
    outside_n = ((test < edges[0]) | (test > edges[-1])).sum()
    assert c.sum() + outside_n == len(test)
    print(name, "유효:", len(test), "빈도:", c.tolist(),
          "범위 밖:", int(outside_n))

<table header-row="true">
<tr><td>사례</td><td>기대 빈도</td><td>해석</td></tr>
<tr><td>0, 20, 40, 60, 80, 100</td><td>1, 1, 1, 1, 2</td><td>100은 마지막 구간에 포함</td></tr>
<tr><td>20, 20, 20</td><td>0, 3, 0, 0, 0</td><td>반복값을 모두 센다</td></tr>
<tr><td>20</td><td>0, 1, 0, 0, 0</td><td>단일값으로 분포 형태를 단정하지 않는다</td></tr>
<tr><td>전부 결측</td><td>0, 0, 0, 0, 0</td><td>유효 자료 없음. 상대빈도 계산을 하지 않는다</td></tr>
<tr><td>10, 20, 150</td><td>1, 1, 0, 0, 0</td><td>150 한 건이 범위 밖. 범위를 확장하거나 제외 사실을 설명한다</td></tr>
</table>

## 재현 가능한 요청


먼저 자신의 손그림을 완성한 뒤 자료 생성 코드를 함께 전달합니다.

```plain text
제공한 raw는 교육용 가상 학생 22명의 편도 통학 응답이며 단위는 분이다.
결측은 2건이다. 원자료를 수정하지 말고 그림에는 시간 유효값 20건을 사용하라.
비음수 정수 조건을 확인하고 줄기=십의 자리, 잎=일의 자리로 출력하라.
반복값과 빈 줄기를 보존하고 2 | 5 = 25분이라는 키를 표시하라.
히스토그램 경계는 [0,20,40,60,80,100], 세로축은 빈도로 하라.
마지막 구간만 오른쪽 끝을 포함하고, 구간 빈도와 범위 밖 개수를 표로 제시하라.
같은 자료로 폭 10·20·25분을 비교하고 시작점 0·5분 비교는 별도로 수행하라.
비교 그림의 축 범위를 맞추고 모든 그림에 단위와 유효 수·결측 수를 설명하라.
손계산과 직접 조건문 집계, 경계값 사례로 검증하라.
계산된 사실과 분포에 대한 추측을 구분하고, 90분을 자동 삭제하지 말라.
실제로 실행하지 않았다면 예상 결과라고 명시하라.
```

## 진단 활동


다음 주장마다 “잘못된 점 → 수정 → 확인 방법”을 한 줄씩 적으세요.

1. “raw.fillna(0)으로 정리했더니 첫 구간이 6명이다.”
2. “hist(x, bins=10)이므로 구간 폭은 10분이다.”
3. “0–60분 그림이 예쁘므로 이것을 전체 분포로 제출한다.”
4. “hist와 histogram의 결과가 같으니 독립적 검증을 마쳤다.”
5. “90분은 떨어져 있으니 삭제한다.”

<details>
<summary>확인용 해설</summary>
	1. 미응답을 0분 관측값으로 만들었습니다. 결측 2건을 따로 보고하고 유효값만 사용하면 첫 구간은 4건입니다.
	2. 정수 bins는 구간 개수입니다. 실제 반환된 경계를 확인하거나 경계 목록을 직접 지정합니다.
	3. 90분이 빠져 19건만 표현됩니다. 전체 범위를 포함하거나 확대 그림임을 밝히고 전체 그림을 함께 제시합니다.
	4. 두 함수는 집계 구현을 공유합니다. 손계산과 직접 조건문으로 따로 확인합니다.
	5. 큰 값이라는 관찰은 삭제 근거가 아닙니다. 단위·입력·수집 상황을 확인하고 원자료를 보존합니다.
</details>

## 보충: 상대빈도와 밀도는 다릅니다


동일한 폭의 이번 실습에서는 빈도 막대로 비교할 수 있습니다. 폭이 다른 구간을 한 그림에 섞을 때 높이를 그대로 빈도로 두면 넓은 막대가 시각적으로 과장될 수 있습니다.
밀도 높이는 **구간 빈도 ÷ (유효 수 × 구간 폭)**입니다. 이때 막대의 **넓이**가 상대빈도이고 전체 넓이는 1입니다. 밀도 높이의 합이 1일 필요는 없습니다. density=True를 “막대 높이가 확률”이라고 설명하면 안 됩니다.
이 단락은 선택 보충입니다. 필수 과제는 같은 폭의 구간으로 수행합니다.

In [ ]:
unequal_edges = np.array([0, 20, 40, 100])
density, _ = np.histogram(x, bins=unequal_edges, density=True)
area = np.sum(density * np.diff(unequal_edges))
assert np.isclose(area, 1)
print("밀도:", density, "넓이 합:", area)

이 계산은 모든 유효값이 지정 범위에 포함된 이번 자료를 전제로 합니다. 범위를 잘라 정규화하면 포함된 값들에 대한 밀도가 되므로 제외된 값을 따로 밝혀야 합니다.

## 권장 예


“이 가상 자료는 전체 22건 중 시간 유효 응답 20건을 사용했다. 20분 폭과 0분 시작점에서 20분 이상 40분 미만 구간에 9건(45%)이 포함된다. 관측값 대부분은 10–60분이고 90분 값이 오른쪽에 떨어져 있다. 폭과 시작점을 바꾸면 봉우리와 빈 구간의 인상은 달라지므로 그림 하나로 모집단 분포나 집단의 존재를 단정할 수 없다. 90분의 오류 여부와 시간 미응답의 이유는 확인되지 않았다.”

## 피해야 할 예


- “우리 학교 학생의 45%는 통학 시간이 20–40분이다.” → 가상 소표본이며 실제 모집단의 비율이 아닙니다.
- “두 봉우리가 보이므로 두 집단이 있다.” → 구간 설정과 표본 변동, 실제 집단 변수를 확인해야 합니다.
- “히스토그램에 없으니 90분은 없다.” → 표시·집계 범위에서 빠졌을 수 있습니다.

## 검증 로그 양식


```plain text
분석 질문·자료 출처·관측 단위:
전체/유효/결측 수:
그림 종류·단위·키:
구간 경계 목록·경계 포함 규칙:
세로축 종류·가로/세로축 표시 범위:
구간 빈도·빈도 합계·범위 밖 개수:
손계산 대조:
다른 계산 경로 대조:
경계값·전부 결측·단일값 실험:
폭과 시작점을 바꿔도 유지되는 관찰:
설정에 따라 달라지는 관찰:
AI 요청 원문·응답·수정 이유:
확인하지 못한 점:
```

## 개인 실습 과제


**제출 기한·방식·배점은 강의 소개와 LMS의 최신 공지를 따릅니다.** 하나의 Colab 노트북에 다음을 담으세요.

1. 자료 사전과 분석 질문, 원자료·전체·유효·결측 수.
2. 키와 단위가 있는 손작성 줄기와 잎 그림 및 코드 재현, 잎 수와 원자료 복원 확인.
3. 20분 폭의 손계산 빈도표·히스토그램·조건문 집계.
4. 폭 10·20·25분 비교 그림 3개와 시작점 0·5분 비교 그림 2개.
5. 경계값·전부 결측·단일값을 포함한 검증 로그.
6. AI 요청 원문·실제 응답·수정 내역과 해석 3–5문장.
7. 새 실행 세션에서 위에서 아래로 재실행한 결과.

자기점검: 자료와 분모가 명확한가, 경계를 재현할 수 있는가, 빈도 합계가 맞는가, 설정 변화와 해석을 연결했는가, 가상 자료의 한계를 밝혔는가?
선택 확장으로 구간 폭이 다른 히스토그램의 넓이 합을 확인하거나, 줄기를 0–4·5–9 잎으로 분할해 가독성을 비교할 수 있습니다. 추가 데이터 수집은 필수가 아닙니다.

## 오늘의 정리와 다음 주


- 줄기와 잎 그림은 키와 정밀도를 명시해야 하며 반복값과 빈 줄기를 확인합니다.
- 히스토그램은 값들을 묶기 때문에 구간 안의 세부 정보를 잃습니다.
- 구간 폭·시작점·범위는 그림과 해석을 바꿉니다.
- 빈도 합계와 경계값을 검증하고 관찰과 추측을 구분합니다.
- 큰 값은 원자료와 맥락을 확인할 출발점입니다.

**다음 주:** 중심위치의 측도와 저항성. 같은 자료의 평균·중앙값을 비교하고, 큰 값 하나가 대표값을 얼마나 바꾸는지 살펴봅니다. 4주차에서 본 모양을 5주차의 수치 요약과 연결합니다.

## 연결 자료와 참고


- <mention-page url="https://app.notion.com/p/3d396f1add76818f87edd9e72d854a93">강의 소개와 주차별 계획</mention-page>
- <mention-page url="https://app.notion.com/p/3d196f1add768148a985d21d826ca92b">1주차: EDA의 현시성과 해석의 한계</mention-page>
- <mention-page url="https://app.notion.com/p/3d396f1add7680af9fbaf004faccfa7c">2주차: 자료 구조와 분석 명세</mention-page>
- [NumPy: histogram](https://numpy.org/doc/stable/reference/generated/numpy.histogram.html): 구간·범위·밀도의 정의.
- [Matplotlib: hist](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.hist.html): 히스토그램 작성과 반환값.
- 주교재: 허명회, 『R을 활용한 탐색적 자료분석』 제3판, 자유아카데미, 2024.
- John W. Tukey, Exploratory Data Analysis, 1977.

강의계획서와 1·2주차 흐름을 바탕으로 구성한 수업 자료입니다. 예제·활동·확인값은 교육 목적으로 작성했으며 실제 학생 통계가 아닙니다.